# 1) Data Preprocessing
In this step I prepare the data to be used for training the neural network.

The code uses a Google Ads search term report as the basis. It is also necessary to get a keyword report of each ad group to compute the average embedding of each ad group.

In [ ]:
!pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
from tqdm import tqdm

In [ ]:
#Open csv search term report
f = "report.csv"
updated_df = pd.read_csv(f)
#Reorganize data to follow order "Search Term", "Keyword", "Added/Excluded"
updated_df = updated_df.iloc[:,[0,3,1,2]].dropna()
updated_df.head()

In [ ]:
# Get rid of match type characers in the keywords
for i in range(0,len(updated_df['Keyword'])):
  keyword = updated_df['Keyword'][i]
  if '"' in keyword:
    keyword = keyword[1:len(keyword)-1]
    updated_df['Keyword'][i] = keyword
  elif "[" in keyword:
    keyword = keyword[1:len(keyword)-1]
    updated_df['Keyword'][i] = keyword

In [ ]:
# Check updated keywords
updated_df.head()

In [ ]:
# Change Added/Excluded column to class numbers
for i in range(0, len(updated_df["Added/Excluded"])):
  value = updated_df["Added/Excluded"][i]
  if value == "Added":
    updated_df["Added/Excluded"][i] = 0
  elif value == "Excluded":
    updated_df["Added/Excluded"][i] = 1
  elif value == "None":
    updated_df["Added/Excluded"][i] = 2
  else:
    print("Couldn't update value")

In [ ]:
# Check mapping is correct
updated_df.head()

In [ ]:
# Check class distribution
updated_df['Added/Excluded'].plot.hist()

In [ ]:
# Load model to compute sentence embeddings
model = SentenceTransformer("all-mpnet-base-v2")

In [ ]:
# Calculate semantic similarity for each row
# The score will go into a new DataFrame that will then be appended to the updated_df DataFrame
scores = {}
value_list = []
for i in tqdm(range(0, len(updated_df["Search term"]))):
  term_embedding = model.encode(updated_df["Search term"][i])
  keyword_embedding = model.encode(updated_df["Keyword"][i])
  value_list.append(util.pytorch_cos_sim(term_embedding, keyword_embedding).item())

In [ ]:
# Check results of embedding
scores["Similarity Score"] = value_list
scores_df = pd.DataFrame.from_dict(scores)
scores_df.head()

In [ ]:
# Append scores_df to updated_df
updated_df = pd.concat([updated_df,scores_df], axis=1)
updated_df.head()

In [ ]:
updated_df = pd.get_dummies(updated_df, prefix="Match Type", columns=["Match type"])

In [ ]:
# Create CSV file of updated report data as process backup
updated_df.to_csv("updated_report_encoded.csv")

In [ ]:
# Load previous file if backup needs to be loaded
updated_df = pd.read_csv("updated_report_encoded.csv")
updated_df = updated_df.drop("Unnamed: 0", axis=1)
updated_df.head()

In [ ]:
# Import os utilities and create a file list
from os import listdir
folder = "ad_groups"
files = listdir(folder)
print(files)

In [ ]:
# Get average embedding for each ad group
ad_group_list = []
for f in files:
  value_list = []
  path = "ad_groups/" + f
  keyword_df = pd.read_csv(path)
  keyword_df = keyword_df[["Keyword"]].copy().dropna()
  for keyword in keyword_df["Keyword"]:
    value_list.append(model.encode(keyword))
  ad_group_list.append(sum(value_list)/len(value_list))

In [ ]:
# Compare each keyword with each ad group average and get highest similarity score

value_list = []
for term in tqdm(updated_df["Search term"]):
  term_embedding = model.encode(term)
  value = 0
  for score in ad_group_list:
    similarity = util.pytorch_cos_sim(term_embedding, score).item()
    if similarity > value:
      value = similarity
  value_list.append(value)

In [ ]:
# Save keyword vs ad group score as DataFrame
value_dict = {}
value_dict["Keyword vs Account Similarity"] = value_list
value_df = pd.DataFrame.from_dict(value_dict)
value_df.head()

In [ ]:
# Merge previous DataFrame into updated_df
updated_df = pd.concat([updated_df, value_df], axis=1)
updated_df = updated_df.iloc[:,[0,1,3,7,4,5,6,2]]
updated_df.head()

In [ ]:
# Create CSV file of updated report data as process backup
updated_df.to_csv("final_report.csv")

In [ ]:
updated_df.corr()

In [ ]:
updated_df["Added/Excluded"].value_counts()

In [ ]:
class_0 = updated_df["Added/Excluded"].value_counts()[0]
class_1 = updated_df["Added/Excluded"].value_counts()[1]
class_2 = updated_df["Added/Excluded"].value_counts()[2]
total = updated_df["Added/Excluded"].value_counts().sum()
print(f"Contribution: 0->{(class_0/total)*100} / 1->{(class_1/total)*100} / 2-> {(class_2/total)*100}")

#2) Decision tree algorithm
In this section we train a Scikit Learn tree algorithm to perform inference on the data set. We will compare the results of this with a neural network to find the best performing technology for the problem.

In [ ]:
# Map "Added", "None", "Excluded" vales as classes
key_map = {
    0:"Added",
    1:"Excluded",
    2:"None"
}

In [ ]:
# Load data backup if session is restarted
updated_df = pd.read_csv("final_report.csv")
updated_df = updated_df.drop(labels=["Unnamed: 0"], axis=1)
updated_df.head()

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score

In [ ]:
# Create a classifier
criterion = "gini"
class_weights = {0: 2, 1: 2, 2:1}
tree_classifier = DecisionTreeClassifier(criterion=criterion, random_state=42)

In [ ]:
# Shrink database to have balanced class distributions
trimmed_df = updated_df.copy()
remove = trimmed_df.index[trimmed_df["Added/Excluded"] == 2]
remove = remove[:14000]
trimmed_df.drop(index=remove, inplace=True)

In [ ]:
trimmed_df["Added/Excluded"].value_counts()

In [ ]:
# Create testing and training data
# "Similarity Score",
X = trimmed_df[["Similarity Score","Keyword vs Account Similarity","Match Type_Broad match","Match Type_Exact match","Match Type_Phrase match"]]
Y = trimmed_df["Added/Excluded"]

In [ ]:
# Create train and test datasets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

In [ ]:
# Train classifier
tree_classifier = tree_classifier.fit(X_train, Y_train)

In [ ]:
# Get model metadata
print("Classifier depth: ", tree_classifier.get_depth())
print("Classifier leaves: ", tree_classifier.get_n_leaves())
print("Classifier params: ", tree_classifier.get_params())

In [ ]:
# Get model inferences for acccuracy scoring
Y_pred = tree_classifier.predict(X_test)
accuracy = accuracy_score(Y_test, Y_pred)
print(accuracy*100)

In [ ]:
# Get model precision
precision = precision_score(Y_test, Y_pred, average="weighted")
print(precision*100)

In [ ]:
# Get feature importance
importance = tree_classifier.feature_importances_
labels = X.columns
importance_df = pd.DataFrame(data=importance, index=labels, columns=["Importance"])
importance_df.head()

In [ ]:
import numpy as np

In [ ]:
# Test the inference with a sample not contained in the network
test_keyword = "roof replacement specialist"
test_search_term = "roof replacement contractor"

keyword_embedding1 = model.encode(test_keyword)
search_term_embedding1 = model.encode(test_search_term)
keyword_similarity = util.pytorch_cos_sim(keyword_embedding1, search_term_embedding1).item()

embedding2 = model.encode(test_search_term)
account_similarity = 0
for value in ad_group_list:
  similarity2 = util.pytorch_cos_sim(search_term_embedding1, value).item()
  if similarity2 > account_similarity:
    account_similarity = similarity2

broad_match = 1
phrase_match = 0
exact_match = 0

test_packet = [keyword_similarity, account_similarity, broad_match, phrase_match, exact_match]
test_packet = np.array(test_packet)
prediction = tree_classifier.predict(test_packet.reshape(1,-1))[0]
print("Predicted class: ", key_map[prediction])

In [ ]:
keyword_word_list = test_keyword.split(" ")
term_word_list = test_search_term.split(" ")
one_to_one_score = []
for word1 in keyword_word_list:
  e1 = model.encode(word1)
  top_sim = 0
  for word2 in term_word_list:
    e2 = model.encode(word2)
    one_to_one_similarity = util.pytorch_cos_sim(e1/keyword_embedding1,e2/search_term_embedding1).item()
    if one_to_one_similarity > top_sim:
      top_sim = one_to_one_similarity
      print(f"Word1: {word1} -- Word2: {word2} -- Similarity: {one_to_one_similarity}")
      one_to_one_score.append(one_to_one_similarity)

print("Complete phrase similarity: ", keyword_similarity)
print("Account similarity: ", account_similarity)
print("W2W similarity: ", sum(one_to_one_score)/len(one_to_one_score))

# 3) Neural Network
In this step, I implement a basic neural network using PyTorch to compare it's performance with the decision tree above.

In [ ]:
import torch
from torch import nn, optim
import torch.nn.init as init
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

In [ ]:
# Get the number of features
in_features = len(X_train.columns)

# Make Scikit Learn datasplits a tensor. Uses np.values to convert the dataframes into NumPy arrays before making them a tensor
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train.values, dtype=torch.long)
Y_test_tensor = torch.tensor(Y_test.values, dtype=torch.long)

#print(X_train_tensor.shape)
#print(Y_train_tensor.shape)
#print(X_test_tensor.shape)
#print(Y_test_tensor.shape)

# Create train and test dataset
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

# Create training and test DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [ ]:
# Define simple neural network
class Network(nn.Module):
  def __init__(self, in_features=5, classes=3):
    super().__init__()
    self.l1 = nn.Linear(in_features, in_features*2)
    self.l2 = nn.Linear(in_features*2, in_features*4)
    self.l3 = nn.Linear(in_features*4, in_features*8)
    self.l4 = nn.Linear(in_features*8, in_features*4)
    self.out = nn.Linear(in_features*4, classes)

    self.relu = nn.ReLU()

    self.initialize_weights()

  def forward(self, x):
    #print(x.shape)
    x = self.relu(self.l1(x))
    #print(x.shape)
    x = self.relu(self.l2(x))
    #print(x.shape)
    x = self.relu(self.l3(x))
    #print(x.shape)
    x = self.relu(self.l4(x))
    #print(x.shape)
    x = nn.Softmax(dim=1)(self.out(x))
    #print(x.shape)
    #print(x)
    return x

  def initialize_weights(self):
    for m in self.modules():
      if isinstance(m, nn.Linear):
        init.xavier_normal_(m.weight)
        if m.bias is not None:
          init.constant_(m.bias, 0)

In [ ]:
# Create model instance and generate random weights
network = Network(in_features, classes=3)

# Define training parameters
epoch = 100
lr = 0.0001
criterion = nn.CrossEntropyLoss() # Error function to minimize
optimizer = optim.Adam(network.parameters(), lr, weight_decay=0.001)

In [ ]:
# Create training loop
train_loss_list = []
test_loss_list = []
train_acc_list = []
test_acc_list = []

for e in tqdm(range(epoch)):
  # Puts network into training mode (that is, it will collect gradient data)
  network.train()
  train_loss = 0
  train_accuracy = 0
  correct = 0
  items = 0

  for inputs, labels in train_dataloader:
    optimizer.zero_grad() # Cleans the grad log
    outputs = network(inputs)
    loss = criterion(outputs, labels)
    loss.backward() # Calculates gradient
    optimizer.step() # Updates weights
    train_loss += loss.item()
    _, predicted = torch.max(outputs, 1) # Gets the class with the biggest probability (1 => dimension 2)
    correct += (predicted == labels).sum().item()
    items += labels.size(0)

  train_accuracy += correct/items*100
  #print(f"Training Loss: {train_loss:.4f}. Train Accuracy: {train_accuracy:.2f}%")
  train_loss_list.append(train_loss)
  train_acc_list.append(train_accuracy)

  val_loss = 0
  val_accuracy = 0
  correct = 0
  items = 0
  network.eval() # Does not track gradients to speed up compute
  for inputs, labels in test_dataloader:
    outputs = network(inputs)
    loss = criterion(outputs, labels)
    val_loss += loss.item()
    _, predicted = torch.max(outputs, 1)
    correct += (predicted == labels).sum().item()
    items += labels.size(0)

  val_accuracy += correct/items*100
  #print(f"Test Loss: {val_loss:.4f}. Test Accuracy: {val_accuracy:.2f}%")
  test_loss_list.append(val_loss)
  test_acc_list.append(val_accuracy)

plt.figure()
plt.subplot(2,1,1)
plt.plot(train_loss_list, label="Training Loss")
plt.plot(test_loss_list, label="Testing Loss")
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.subplot(2,1,2)
plt.plot(train_acc_list, label="Training Accuracy")
plt.plot(test_acc_list, label="Testing Accuracy")
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

print(f"Max training accuracy: {max(train_acc_list)}% ----- Max testing accuracy: {max(test_acc_list)}")

In [ ]:
# Perform inference with the network and the same test packet used in the tree test
test_packet = torch.tensor(test_packet, dtype=torch.float32)
print("Test Packet: ", test_packet)
print("Test Packet Shape: ", test_packet.shape)
prediction = network(test_packet.unsqueeze(0)) # .unsqueeze adds a batch dimension because softmax needs a 2d tensor
class_prediction = torch.argmax(prediction).item()
print("Predictions: ", prediction)
print("Predicted Class: ", key_map[class_prediction])
print("Prediction Shape: ", prediction.shape)